<a href="https://colab.research.google.com/github/lynnkaram/health-informatics-projects/blob/main/homework-3-clinical-text-deidentification/clinical_text_deidentification_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Installation of PyDeid**

In [ ]:
!pip install -q git+https://github.com/GEMINI-Medicine/pyDeid
!pip install -q Faker


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.0 MB/s eta 0:00:00


In [ ]:
# Example
from pyDeid import deid_string
phi, new_text = deid_string('Elijah Wood starred in The Lord of the Rings, released on 10 December 2001.')

print(phi)      # what PyDeid detected (names, dates, etc.)
print(new_text) # the de-identified sentence

[{'phi_start': 0, 'phi_end': 6, 'phi': 'Elijah', 'surrogate_start': 0, 'surrogate_end': 4, 'surrogate': 'Ryan', 'types': ['Male First Name (un)', 'Last Name (un)', 'First Name4 (NamePattern1)']}, {'phi_start': 7, 'phi_end': 11, 'phi': 'Wood', 'surrogate_start': 5, 'surrogate_end': 13, 'surrogate': 'Saunders', 'types': ['Last Name (ambig)', 'Last Name (NamePattern1)']}, {'phi_start': 58, 'phi_end': 74, 'phi': '10 December 2001', 'surrogate_start': 60, 'surrogate_end': 70, 'surrogate': '1995-11-19', 'types': ['Day Month [dd of Month]', 'Day Month Year [dd-Month-yy(yy)]', 'Month Year [Month of yy(yy)]', 'Day Month Year (2) [dd of Month, yy(yy)]']}]
Ryan Saunders starred in The Lord of the Rings, released on 1995-11-19.


**De-identifying Clinical Texts**

In [ ]:
!pip install -q pdfplumber requests

import io, re, requests, pdfplumber
from pyDeid import deid_string

def strip_newlines(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

# Text 1
pdf_url = "https://dss.mo.gov/mhd/cs/psych/pdf/progressnote_indv_sample.pdf"
start_marker = "Recipient Information"
end_marker   = "ground her further."

bin_pdf = requests.get(pdf_url, timeout=60).content
with pdfplumber.open(io.BytesIO(bin_pdf)) as pdf:
    raw = "\n".join((p.extract_text() or "") for p in pdf.pages)

one_line = strip_newlines(raw)
i = one_line.find(start_marker)
j = one_line.find(end_marker)
assert i != -1 and j != -1, "Span not found—check markers."
text1 = one_line[i : j + len(end_marker)]
phi1, deid1 = deid_string(text1)
print(deid1)


Recipient Information Provider Information Name: Samantha Martinez Name: Thumb Psychological Assoc. Medicaid Number: 12345678 Medicaid Number: 367-684-492 Number of Group Members (GT only): Therapist Name: Tom Thumb, Ph.D., Licensed Psychologist Family Members Present (FT only): Date of Service: 19th of April, 1986 Begin & End Time: 7:59:00 p.m.–8:49:00 p.m. Type of Service : Individual Therapy 90806 Service Setting: Office Sample Progress Note Patient report of recent symptoms/behaviors: (R/T DX & TX Plan) Samantha denied any suicidal ideation in the past week. She reported that she still feels sad most of the time. She got an “F” on another math test this week. She expressed frustration with math. She was tearful as she talked about feeling dumb and feeling like she will never understand it. However, she reported that when she got the “F” her father was more understanding, did not lecture her or ground her further.


In [ ]:
# Text 2
url2 = "https://svn.apache.org/repos/asf/ctakes/branches/ctakes-3.2.3/ctakes-clinical-pipeline/src/test/data/plaintext/testpatient_plaintext_1.txt"
orig2 = strip_newlines(requests.get(url2, timeout=30).text)
phi2, deid2 = deid_string(orig2)
print(deid2)


Dr. Anderson Medical Nutrition Therapy for Hyperlipidemia Referral from: William Rodriguez, RD, LD, CNSD Phone contact: 647-453-4650 Height: 144 cm Current Weight: 45 kg Date of current weight: 02-29-2001 Admit Weight: 53 kg BMI: 18 kg/m2 Diet: General Daily Calorie needs (kcals): 1500 calories, assessed as HB + 20% for activity. Daily Protein needs: 40 grams, assessed as 1.0 g/kg. Pt has been on a 3-day calorie count and has had an average intake of 1100 calories. She was instructed to drink 2-3 cans of liquid supplement to help promote weight gain. She agrees with the plan and has my number for further assessment. May want a Resting Metabolic Rate as well. She takes an aspirin a day for knee pain.


In [ ]:
# Text 3
url3 = "https://svn.apache.org/repos/asf/ctakes/branches/ctakes-3.2.3/ctakes-clinical-pipeline/src/test/data/plaintext/testpatient_plaintext_2.txt"
orig3 = strip_newlines(requests.get(url3, timeout=30).text)
phi3, deid3 = deid_string(orig3)
print(deid3)

Dr. Smith Medical Nutrition Therapy for Hyperlipidemia Significant for her father, who died at 50 years from colon cancer. Phone contact: 431-120-8340 Height: 144 cm Current Weight: 45 kg Diet: General Daily Calorie needs (kcals): 1500 calories, assessed as HB + 20% for activity. Daily Protein needs: 40 grams, assessed as 1.0 g/kg. Colonoscopy. No pain or discomfort reported.
